In [ ]:
!pip install x-transformers pythainlp gensim matplotlib --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.8/99.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.8/104.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.4 MB/s eta 0:00:00


In [ ]:
RANDOM_STATE = 42
BATCH_SIZE = 128
LR = 2e-4
EPOCHS = 5

HIDDEN_DIM = 300
DEPTH = 4
HEADS = 4
MAX_LEN = 512
MAX_VOCAB = 20000

PAD_ID = 0
UNK_ID = 1

In [ ]:
labels = [
    "politics", "human_rights", "quality_of_life", "international",
    "social", "environment", "economics", "culture", "labor",
    "national_security", "ict", "education"
]

id_to_label = {i:l for i,l in enumerate(labels)}
NUM_CLASSES = len(labels)

In [ ]:
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, classification_report
from collections import Counter

from x_transformers import TransformerWrapper, Decoder
from pythainlp.tokenize import word_tokenize
from pythainlp.util import normalize
from pythainlp.word_vector import WordVector

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
path = "/content/"

In [ ]:
train_df = pd.read_csv(f"{path}/prachatai_train.csv", encoding='latin1')
val_df   = pd.read_csv(f"{path}/prachatai_validation.csv", encoding='latin1')
test_df  = pd.read_csv(f"{path}/prachatai_test.csv", encoding='latin1')

print("Train DataFrame columns:", train_df.columns)
print("Val DataFrame columns:", val_df.columns)
print("Test DataFrame columns:", test_df.columns)

Train DataFrame columns: Index(['Unnamed: 0', 'url', 'date', 'title', 'body_text', 'politics',
       'human_rights', 'quality_of_life', 'international', 'social',
       'environment', 'economics', 'culture', 'labor', 'national_security',
       'ict', 'education'],
      dtype='object')
Val DataFrame columns: Index(['Unnamed: 0', 'url', 'date', 'title', 'body_text', 'politics',
       'human_rights', 'quality_of_life', 'international', 'social',
       'environment', 'economics', 'culture', 'labor', 'national_security',
       'ict', 'education'],
      dtype='object')
Test DataFrame columns: Index(['Unnamed: 0', 'url', 'date', 'title', 'body_text', 'politics',
       'human_rights', 'quality_of_life', 'international', 'social',
       'environment', 'economics', 'culture', 'labor', 'national_security',
       'ict', 'education'],
      dtype='object')


In [ ]:
def process_th(text):
    text = normalize(str(text))
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def tokenize_th(text):
    tokens = word_tokenize(text, engine="newmm", keep_whitespace=False)
    return [t for t in tokens if len(t.strip()) > 1]

train_df['text'] = train_df['body_text'].apply(process_th).apply(tokenize_th)
val_df['text']   = val_df['body_text'].apply(process_th).apply(tokenize_th)
test_df['text']  = test_df['body_text'].apply(process_th).apply(tokenize_th)


In [ ]:
counter = Counter()
for t in train_df['text']:
    counter.update(t)

vocab = {"<pad>": PAD_ID, "<unk>": UNK_ID}
for i, (w, _) in enumerate(counter.most_common(MAX_VOCAB - 2), start=2):
    vocab[w] = i

def encode(tokens):
    if len(tokens) > MAX_LEN:
        half = MAX_LEN // 2
        tokens = tokens[:half] + tokens[-half:]
    ids = [vocab.get(t, UNK_ID) for t in tokens]
    return ids + [PAD_ID] * (MAX_LEN - len(ids))

vocab_size = len(vocab)
print("Loading Thai2Vec...")
thai2vec = WordVector()

In [ ]:
embedding_weight = np.zeros((vocab_size, HIDDEN_DIM), dtype=np.float32)
rng = np.random.default_rng(RANDOM_STATE)
embedding_weight[UNK_ID] = rng.normal(0, 0.01, size=(HIDDEN_DIM,))

for word, idx in vocab.items():
    if word in ("<pad>", "<unk>"):
        continue
    try:
        embedding_weight[idx] = thai2vec.get_vector(word)
    except:
        embedding_weight[idx] = rng.normal(0, 0.01, size=(HIDDEN_DIM,))

In [ ]:
class TextDataset(Dataset):
    def __init__(self, df):
        self.texts = df['text'].tolist()
        self.labels = df['label'].values

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return (
            torch.tensor(encode(self.texts[idx]), dtype=torch.long),
            torch.tensor(int(self.labels[idx]), dtype=torch.long)
        )

train_loader = DataLoader(TextDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TextDataset(val_df), batch_size=BATCH_SIZE)
test_loader  = DataLoader(TextDataset(test_df), batch_size=BATCH_SIZE)


In [ ]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.transformer = TransformerWrapper(
            num_tokens=vocab_size,
            max_seq_len=MAX_LEN,
            attn_layers=Decoder(dim=HIDDEN_DIM, depth=DEPTH, heads=HEADS)
        )

        W = torch.tensor(embedding_weight, dtype=torch.float32)
        for m in self.transformer.modules():
            if isinstance(m, nn.Embedding) and m.weight.shape == W.shape:
                m.weight.data.copy_(W)
                m.weight.data[PAD_ID].zero_()

        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(HIDDEN_DIM, NUM_CLASSES)

    def forward(self, x):
        mask = (x != PAD_ID)
        emb = self.transformer(x, mask=mask, return_embeddings=True)

        mask = mask.unsqueeze(-1).float()
        pooled = (emb * mask).sum(1) / mask.sum(1).clamp(min=1)

        return self.fc(self.dropout(pooled))


In [ ]:
model = Model().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

@torch.no_grad()
def evaluate(loader):
    model.eval()
    preds, labels_ = [], []

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        out = model(x).argmax(1)
        preds.append(out.cpu().numpy())
        labels_.append(y.cpu().numpy())

    y_pred = np.concatenate(preds)
    y_true = np.concatenate(labels_)

    acc = (y_pred == y_true).mean()
    f1_micro = f1_score(y_true, y_pred, average="micro")
    f1_macro = f1_score(y_true, y_pred, average="macro")
    f1_weighted = f1_score(y_true, y_pred, average="weighted")

    return acc, f1_micro, f1_macro, f1_weighted, y_true, y_pred

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    val_acc, f1_micro, f1_macro, f1_weighted, _, _ = evaluate(val_loader)

    print(
        f"Epoch {epoch+1:03d} | "
        f"loss={total_loss:.6f} | "
        f"acc={val_acc:.6f} ({val_acc*100:.2f}%) | "
        f"f1_micro={f1_micro:.6f} | "
        f"f1_macro={f1_macro:.6f} | "
        f"f1_weighted={f1_weighted:.6f}"
    )

In [ ]:
test_acc, f1_micro, f1_macro, f1_weighted, y_true, y_pred = evaluate(test_loader)

print(
    f"\nFINAL MODEL TEST:\n"
    f"Accuracy     = {test_acc:.6f} ({test_acc*100:.2f}%)\n"
    f"F1-micro     = {f1_micro:.6f}\n"
    f"F1-macro     = {f1_macro:.6f}\n"
    f"F1-weighted  = {f1_weighted:.6f}"
)

print("\n===== CLASSIFICATION REPORT =====")
print(classification_report(y_true, y_pred, target_names=labels))

# =========================
# PREDICT
# =========================
@torch.no_grad()
def predict(text):
    tokens = tokenize_th(process_th(text))
    x = torch.tensor([encode(tokens)], dtype=torch.long).to(DEVICE)

    probs = torch.softmax(model(x), dim=1).cpu().numpy()[0]
    pred = int(np.argmax(probs))

    return {
        "label_id": pred,
        "label_name": id_to_label[pred],
        "confidence": float(probs[pred])
    }

print(predict("รัฐบาลประกาศนโยบายเศรษฐกิจใหม่"))